# Analisis Domisili & Polis

Notebook ini mengeksplorasi distribusi domisili per polis, statistik polis, dan hubungannya dengan data klaim.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100

In [ ]:
data_basepath = 'dataset'

data_polis = pd.read_csv(f'{data_basepath}/Data_Polis.csv')

# parse tanggal (format YYYYMMDD sebagai integer)
for col in ['Tanggal Lahir', 'Tanggal Efektif Polis']:
    data_polis[col] = pd.to_datetime(data_polis[col].astype(str), format='%Y%m%d', errors='coerce')

data_klaim = pd.read_csv(
    f'{data_basepath}/Data_Klaim.csv',
    parse_dates=['Tanggal Pembayaran Klaim', 'Tanggal Pasien Masuk RS', 'Tanggal Pasien Keluar RS']
)

print(f'Data Polis  : {data_polis.shape[0]:,} baris, {data_polis.shape[1]} kolom')
print(f'Data Klaim  : {data_klaim.shape[0]:,} baris, {data_klaim.shape[1]} kolom')
data_polis.head()

## 1. Overview Data Polis

In [ ]:
today = pd.Timestamp.today()
data_polis['Usia'] = ((today - data_polis['Tanggal Lahir']).dt.days / 365.25).round(1)
data_polis['Lama Polis (thn)'] = ((today - data_polis['Tanggal Efektif Polis']).dt.days / 365.25).round(1)

print('=== Info Dataset Polis ===')
print(f'Jumlah polis unik   : {data_polis["Nomor Polis"].nunique():,}')
print(f'Jumlah domisili     : {data_polis["Domisili"].nunique()}')
print(f'Jumlah plan code    : {data_polis["Plan Code"].nunique()}')
print(f'Missing values      :')
print(data_polis.isnull().sum()[data_polis.isnull().sum() > 0] if data_polis.isnull().sum().sum() > 0 else '  (tidak ada)')
print(f'\nStatistik Usia:')
print(data_polis['Usia'].describe().map(lambda x: f'{x:.1f}'))
print(f'\nStatistik Lama Polis (tahun):')
print(data_polis['Lama Polis (thn)'].describe().map(lambda x: f'{x:.1f}'))

## 2. Distribusi Domisili — Count & Proporsi

In [ ]:
domisili_count = data_polis['Domisili'].value_counts().reset_index()
domisili_count.columns = ['Domisili', 'Jumlah Polis']
domisili_count['Persentase (%)'] = (domisili_count['Jumlah Polis'] / domisili_count['Jumlah Polis'].sum() * 100).round(2)
domisili_count['Kumulatif (%)'] = domisili_count['Persentase (%)'].cumsum().round(2)

from IPython.display import display
print('=== Tabel Count & Proporsi Domisili ===')
display(domisili_count.style
    .bar(subset=['Jumlah Polis'], color='#4C72B0')
    .bar(subset=['Persentase (%)'], color='#DD8452')
    .format({'Jumlah Polis': '{:,}', 'Persentase (%)': '{:.2f}%', 'Kumulatif (%)': '{:.2f}%'}))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart — semua domisili
colors = sns.color_palette('viridis', len(domisili_count))
bars = axes[0].barh(domisili_count['Domisili'][::-1],
                     domisili_count['Jumlah Polis'][::-1],
                     color=colors[::-1])
axes[0].set_title('Jumlah Polis per Domisili', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Jumlah Polis')
for bar, val in zip(bars, domisili_count['Jumlah Polis'][::-1]):
    axes[0].text(bar.get_width() + 10, bar.get_y() + bar.get_height()/2,
                 f'{val:,}', va='center', fontsize=8)

# Pie chart — top 6 + others
top6 = domisili_count.head(6).copy()
others_val = domisili_count.iloc[6:]['Jumlah Polis'].sum()
pie_data = pd.concat([top6, pd.DataFrame([{'Domisili': 'Lainnya', 'Jumlah Polis': others_val}])], ignore_index=True)
axes[1].pie(pie_data['Jumlah Polis'], labels=pie_data['Domisili'],
             autopct='%1.1f%%', startangle=140, explode=[0.05]*len(pie_data),
             colors=sns.color_palette('tab10', len(pie_data)))
axes[1].set_title('Proporsi Domisili (Top 6 + Lainnya)', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

## 3. Statistik Polis per Domisili

In [ ]:
stats_domisili = data_polis.groupby('Domisili').agg(
    Jumlah_Polis=('Nomor Polis', 'count'),
    Usia_Rata=('Usia', 'mean'),
    Usia_Min=('Usia', 'min'),
    Usia_Max=('Usia', 'max'),
    Usia_Median=('Usia', 'median'),
    Lama_Polis_Rata=('Lama Polis (thn)', 'mean'),
    Pria=('Gender', lambda x: (x == 'M').sum()),
    Wanita=('Gender', lambda x: (x == 'F').sum()),
).round(1).sort_values('Jumlah_Polis', ascending=False)

stats_domisili['Rasio P/W'] = (stats_domisili['Pria'] / stats_domisili['Wanita']).round(2)

print('=== Statistik Polis per Domisili ===')
display(stats_domisili.style
    .background_gradient(subset=['Jumlah_Polis'], cmap='Blues')
    .background_gradient(subset=['Usia_Rata'], cmap='YlOrRd')
    .format({'Usia_Rata': '{:.1f}', 'Usia_Min': '{:.0f}',
              'Usia_Max': '{:.0f}', 'Usia_Median': '{:.0f}',
              'Lama_Polis_Rata': '{:.1f}', 'Rasio P/W': '{:.2f}'}))

## 4. Distribusi Usia per Domisili (Top 6)

In [ ]:
top6_domisili = domisili_count.head(6)['Domisili'].tolist()
df_top6 = data_polis[data_polis['Domisili'].isin(top6_domisili)].copy()

fig, ax = plt.subplots(figsize=(13, 6))
sns.boxplot(data=df_top6, x='Domisili', y='Usia', order=top6_domisili,
             palette='Set2', ax=ax, width=0.5)
sns.stripplot(data=df_top6, x='Domisili', y='Usia', order=top6_domisili,
               color='gray', alpha=0.2, size=2, ax=ax, jitter=True)

ax.set_title('Distribusi Usia Tertanggung per Domisili (Top 6)', fontsize=13, fontweight='bold')
ax.set_xlabel('Domisili')
ax.set_ylabel('Usia (tahun)')

for i, dom in enumerate(top6_domisili):
    mean_usia = df_top6[df_top6['Domisili'] == dom]['Usia'].mean()
    ax.text(i, mean_usia + 1.5, f'{mean_usia:.1f}', ha='center', fontsize=9,
             color='darkred', fontweight='bold')

plt.tight_layout()
plt.show()

## 5. Distribusi Gender per Domisili

In [ ]:
gender_dom = data_polis.groupby(['Domisili', 'Gender']).size().unstack(fill_value=0)
gender_dom.columns = ['Wanita (F)', 'Pria (M)']
gender_dom = gender_dom.loc[domisili_count['Domisili']]
gender_dom_pct = gender_dom.div(gender_dom.sum(axis=1), axis=0) * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

gender_dom.plot(kind='barh', stacked=True, ax=axes[0],
                 color=['#DD8452', '#4C72B0'], width=0.7)
axes[0].set_title('Jumlah Polis per Gender & Domisili', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Jumlah Polis')
axes[0].set_ylabel('Domisili')
axes[0].legend(loc='lower right')
axes[0].invert_yaxis()

gender_dom_pct.plot(kind='barh', stacked=True, ax=axes[1],
                     color=['#DD8452', '#4C72B0'], width=0.7)
axes[1].set_title('Proporsi Gender per Domisili (%)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Persentase (%)')
axes[1].set_ylabel('')
axes[1].axvline(50, color='white', linestyle='--', linewidth=1.2)
axes[1].legend(loc='lower right')
axes[1].invert_yaxis()
for i, (idx, row) in enumerate(gender_dom_pct.iterrows()):
    axes[1].text(row['Wanita (F)']/2, i, f'{row["Wanita (F)"]:.0f}%',
                  va='center', ha='center', fontsize=7, color='white', fontweight='bold')
    axes[1].text(row['Wanita (F)'] + row['Pria (M)']/2, i, f'{row["Pria (M)"]:.0f}%',
                  va='center', ha='center', fontsize=7, color='white', fontweight='bold')

plt.tight_layout()
plt.show()

## 6. Distribusi Plan Code per Domisili

In [ ]:
plan_dom = data_polis.groupby(['Domisili', 'Plan Code']).size().unstack(fill_value=0)
plan_dom = plan_dom.loc[domisili_count['Domisili']]
plan_dom_pct = plan_dom.div(plan_dom.sum(axis=1), axis=0) * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

plan_dom.plot(kind='barh', stacked=True, ax=axes[0],
               color=sns.color_palette('Set1', 3), width=0.7)
axes[0].set_title('Jumlah Polis per Plan Code & Domisili', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Jumlah Polis')
axes[0].set_ylabel('Domisili')
axes[0].invert_yaxis()

plan_dom_pct.plot(kind='barh', stacked=True, ax=axes[1],
                   color=sns.color_palette('Set1', 3), width=0.7)
axes[1].set_title('Proporsi Plan Code per Domisili (%)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Persentase (%)')
axes[1].set_ylabel('')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

print('\nCrosstab Plan Code vs Domisili (%):')
display(plan_dom_pct.round(1).style.background_gradient(cmap='Blues'))

## 7. Tren Efektif Polis per Domisili

In [ ]:
data_polis['Tahun Efektif'] = data_polis['Tanggal Efektif Polis'].dt.year
trend_dom = data_polis.groupby(['Tahun Efektif', 'Domisili']).size().unstack(fill_value=0)
trend_dom_top6 = trend_dom[top6_domisili]

fig, ax = plt.subplots(figsize=(13, 6))
for col in trend_dom_top6.columns:
    ax.plot(trend_dom_top6.index, trend_dom_top6[col], marker='o', linewidth=2, label=col)

ax.set_title('Tren Jumlah Polis Baru per Tahun (Top 6 Domisili)', fontsize=13, fontweight='bold')
ax.set_xlabel('Tahun Efektif')
ax.set_ylabel('Jumlah Polis')
ax.legend(title='Domisili', bbox_to_anchor=(1.01, 1), loc='upper left')
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
plt.tight_layout()
plt.show()

## 8. Join Klaim — Profil Klaim per Domisili

In [ ]:
klaim_dom = data_klaim.merge(
    data_polis[['Nomor Polis', 'Domisili', 'Plan Code', 'Gender', 'Usia']],
    on='Nomor Polis', how='left'
)

print(f'Total klaim setelah join : {klaim_dom.shape[0]:,}')
print(f'Klaim tanpa domisili     : {klaim_dom["Domisili"].isnull().sum()}')

klaim_per_domisili = klaim_dom.groupby('Domisili').agg(
    Jumlah_Klaim=('Claim ID', 'count'),
    Total_Nominal=('Nominal Klaim Yang Disetujui', 'sum'),
    Rata_Nominal=('Nominal Klaim Yang Disetujui', 'mean'),
    Median_Nominal=('Nominal Klaim Yang Disetujui', 'median'),
).round(0).sort_values('Jumlah_Klaim', ascending=False)

klaim_per_domisili = klaim_per_domisili.join(
    stats_domisili[['Jumlah_Polis']], how='left'
)
klaim_per_domisili['Klaim per Polis'] = (
    klaim_per_domisili['Jumlah_Klaim'] / klaim_per_domisili['Jumlah_Polis']
).round(2)

print('\n=== Profil Klaim per Domisili ===')
display(klaim_per_domisili.style
    .background_gradient(subset=['Jumlah_Klaim'], cmap='Blues')
    .background_gradient(subset=['Klaim per Polis'], cmap='YlOrRd')
    .format({
        'Total_Nominal': 'Rp {:,.0f}',
        'Rata_Nominal': 'Rp {:,.0f}',
        'Median_Nominal': 'Rp {:,.0f}',
    }))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

kpd = klaim_per_domisili.reset_index()

sns.barplot(data=kpd, y='Domisili', x='Jumlah_Klaim', ax=axes[0],
             palette='Blues_r', order=kpd['Domisili'])
axes[0].set_title('Jumlah Klaim per Domisili', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Jumlah Klaim')
for bar, val in zip(axes[0].patches, kpd['Jumlah_Klaim']):
    axes[0].text(bar.get_width() + 2, bar.get_y() + bar.get_height()/2,
                  f'{val:,}', va='center', fontsize=8)

kpd_sorted = kpd.sort_values('Klaim per Polis', ascending=False)
colors_kpp = ['#d62728' if v > kpd['Klaim per Polis'].mean() else '#4C72B0'
               for v in kpd_sorted['Klaim per Polis']]
axes[1].barh(kpd_sorted['Domisili'][::-1], kpd_sorted['Klaim per Polis'][::-1],
              color=colors_kpp[::-1])
axes[1].set_title('Rasio Klaim per Polis per Domisili', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Klaim / Polis')
axes[1].axvline(kpd['Klaim per Polis'].mean(), color='orange', linestyle='--',
                  label=f'Rata-rata: {kpd["Klaim per Polis"].mean():.2f}')
axes[1].legend()
for bar, val in zip(axes[1].patches, kpd_sorted['Klaim per Polis'][::-1]):
    axes[1].text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                  f'{val:.2f}', va='center', fontsize=8)

plt.tight_layout()
plt.show()

## 9. Heatmap Metrik Klaim per Domisili

In [ ]:
pivot_metric = klaim_per_domisili[['Jumlah_Klaim', 'Rata_Nominal', 'Median_Nominal', 'Klaim per Polis']].copy()
pivot_norm = (pivot_metric - pivot_metric.min()) / (pivot_metric.max() - pivot_metric.min())

annot_fmt = pivot_metric.applymap(
    lambda x: f'{x/1e6:.1f}M' if x > 1e4 else f'{x:.2f}'
)

fig, ax = plt.subplots(figsize=(9, 8))
sns.heatmap(pivot_norm, annot=annot_fmt, fmt='', cmap='YlOrRd',
             linewidths=0.5, ax=ax,
             cbar_kws={'label': 'Nilai Ternormalisasi (0-1)'})
ax.set_title('Heatmap Metrik Klaim per Domisili\n(nilai dinormalisasi)', fontsize=12, fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('Domisili')
plt.tight_layout()
plt.show()

## 10. Ringkasan & Insight

In [ ]:
print('=' * 65)
print('  RINGKASAN ANALISIS DOMISILI & POLIS')
print('=' * 65)

top1 = domisili_count.iloc[0]
print(f"\n[Distribusi Polis]")
print(f"  Domisili terbanyak : {top1['Domisili']} ({top1['Jumlah Polis']:,} polis, {top1['Persentase (%)']:.1f}%)")
top2_share = domisili_count.head(2)['Persentase (%)'].sum()
print(f"  Top 2 domisili mendominasi {top2_share:.1f}% dari total polis")

print(f"\n[Profil Tertanggung]")
print(f"  Rata-rata usia : {data_polis['Usia'].mean():.1f} tahun")
print(f"  Usia termuda   : {data_polis['Usia'].min():.0f} tahun")
print(f"  Usia tertua    : {data_polis['Usia'].max():.0f} tahun")
print(f"  Gender pria    : {(data_polis['Gender']=='M').sum():,} ({(data_polis['Gender']=='M').mean()*100:.1f}%)")
print(f"  Gender wanita  : {(data_polis['Gender']=='F').sum():,} ({(data_polis['Gender']=='F').mean()*100:.1f}%)")

print(f"\n[Plan Code]")
for plan, cnt in data_polis['Plan Code'].value_counts().items():
    print(f"  {plan} : {cnt:,} polis ({cnt/len(data_polis)*100:.1f}%)")

print(f"\n[Klaim per Domisili]")
high_ratio = klaim_per_domisili['Klaim per Polis'].idxmax()
low_ratio  = klaim_per_domisili['Klaim per Polis'].idxmin()
highest_avg = klaim_per_domisili['Rata_Nominal'].idxmax()
print(f"  Rasio klaim/polis TERTINGGI : {high_ratio} ({klaim_per_domisili.loc[high_ratio,'Klaim per Polis']:.2f})")
print(f"  Rasio klaim/polis TERENDAH  : {low_ratio} ({klaim_per_domisili.loc[low_ratio,'Klaim per Polis']:.2f})")
print(f"  Rata-rata nominal TERTINGGI : {highest_avg} (Rp {klaim_per_domisili.loc[highest_avg,'Rata_Nominal']:,.0f})")
print('=' * 65)